# Instrumental Variables

**🌐 Language:** **English** | [한국어 →](/instrumental-variables-ko)

<small><em>Written by Myungseok Gwon · <a href="https://github.com/myeongseok-gwon">GitHub</a> · <a href="https://www.linkedin.com/in/myeongseok-gwon/?skipRedirect=true">LinkedIn</a></em></small>

<small><em>Causal inference peer-learning material · Companion video: <a href="https://www.youtube.com/@CausalStudio">Causal Studio</a></em></small>

This notebook is the written-and-coded counterpart of a 7-minute companion video.

Prerequisites:
- Linear regression (OLS)
- Why random assignment (RCTs) gives a clean estimate of a causal effect
- The intuition that simple mean differences can be biased

We build on those by asking: when random assignment is *not* possible, can we use some other randomness already lying around in the real world to identify a causal effect? That's the **instrumental variable**.

## 1) The Snow story: an accident produced by nature

When cholera devastated London in 1854, the physician John Snow noticed that death rates varied wildly from street to street. At the time, two companies piped water into London homes — Lambeth, which drew clean water from upstream, and Southwark & Vauxhall, which drew sewage-mixed water from the lower Thames.

Crucially, **which company served which street had almost nothing to do with the household's income or hygiene**. It was a matter of which pipes had been laid decades earlier.

Snow exploited that accident. He couldn't randomly split households into two groups to decide who drank clean water, but **nature had already produced an equivalent random split** on his behalf.

An instrumental variable (IV) is exactly this kind of tool — one that catches accidents already present in the real world and uses them to pin down a causal effect.

## 2) Notation

We need three variables:

- $Z$ : the **instrument** — a random assignment produced by the experimenter or by nature. (e.g., tablet assignment, lottery number)
- $T$ : the **treatment actually taken**. (e.g., whether the school actually used the tablets)
- $Y$ : the **outcome** we care about. (e.g., test scores)

Plus potential outcomes:

- $Y_0$ : the potential outcome under no treatment
- $Y_1$ : the potential outcome under treatment

The previous chapter introduced the average treatment effect, $ATE = E[Y_1 - Y_0]$. This chapter's main quantity is **an effect restricted to a particular subgroup (the compliers)** — the **LATE**.

## 3) Non-compliance breaks the naive comparison

Take the tablet RCT from the previous chapter. Researchers randomly split the schools into two groups and "assigned" one group to receive tablets while the other served as the control. In the real world, two things happen at once.

1. **Some assigned-treatment schools refuse the tablets** ("the teachers don't want them", "nowhere to store them").
2. **Some assigned-control schools buy tablets with their own budget** ("we're going to use them anyway").

If we now compare "actually treated" vs. "actually untreated", the two groups are no longer comparable. Refusers are probably skeptical of new tech; self-buyers are probably more motivated to begin with.

**Assignment ($Z$) is random, but actual treatment ($T$) is not.** A small simulation makes this concrete.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 4000

# Each school has a latent compliance type: 60% compliers, 20% always-takers, 20% never-takers.
compliance_type = rng.choice(
    ["complier", "always_taker", "never_taker"],
    size=n, p=[0.60, 0.20, 0.20],
)

# Random assignment: half get assigned to treatment.
Z = rng.binomial(1, 0.5, n)

# Actual treatment T depends on type AND assignment.
T = np.where(
    compliance_type == "complier", Z,                  # complier: assignment = treatment
    np.where(compliance_type == "always_taker", 1, 0)   # always / never
)

# Outcome Y: true treatment effect = +5. But baseline scores differ by type.
baseline = np.where(
    compliance_type == "complier", 70,
    np.where(compliance_type == "always_taker", 80, 60)
)
true_effect = 5  # true LATE
Y = baseline + true_effect * T + rng.normal(0, 5, n)

df = pd.DataFrame(dict(Z=Z, T=T, Y=Y, type=compliance_type))
df.head()

,Z,T,Y,type
0,1,1,68.074178,complier
1,0,1,88.706656,always_taker
2,1,1,68.704090,complier
3,0,0,71.881550,complier
4,1,1,70.147837,complier


In [2]:
# Naive comparison: actually-treated vs actually-untreated
naive_diff = df.loc[df["T"] == 1, "Y"].mean() - df.loc[df["T"] == 0, "Y"].mean()
print(f"Naive mean difference (by actual T): {naive_diff:+.2f}")
print(f"True treatment effect:                {true_effect:+.2f}")

Naive mean difference (by actual T): +12.94
True treatment effect:                +5.00


The naive difference is far from the true effect of +5. Why? The actually-treated pool is rich in always-takers (baseline 80) and the actually-untreated pool is rich in never-takers (baseline 60). **Type composition leaks into the mean difference** and becomes bias.

What if we compare by assignment ($Z$) instead?

In [3]:
itt = df.loc[df["Z"] == 1, "Y"].mean() - df.loc[df["Z"] == 0, "Y"].mean()
print(f"ITT (assignment) difference: {itt:+.2f}")

ITT (assignment) difference: +2.63


The intention-to-treat difference lands somewhere around +3 — unbiased, but **smaller than the true effect**, because only some of the schools (the compliers) actually moved with the assignment.

IV's core idea: **divide this small assignment-driven difference by the share of people the assignment actually moved** to recover the effect on one mover.

## 4) Four types: always-taker, never-taker, complier, defier

There are four ways assignment $Z$ can move people, organized by type. By assumption, the last type (defiers) does **not** exist (monotonicity).

| Type | $T$ when $Z = 1$ | $T$ when $Z = 0$ | Note |
|---|:---:|:---:|---|
| **Always-taker** | 1 | 1 | ignores assignment |
| **Never-taker** | 0 | 0 | ignores assignment |
| **Complier** | 1 | 0 | follows assignment |
| **Defier** | 0 | 1 | assumed not to exist (monotonicity) |

IV recovers an effect for **the compliers only**. Always-takers and never-takers behave the same in both arms and contribute nothing to the difference, and defiers are assumed not to exist. This is the **Local Average Treatment Effect (LATE)**.

## 5) The Wald estimator: LATE in numbers

$$
\text{LATE} \;=\; \frac{E[Y \mid Z=1] - E[Y \mid Z=0]}{E[T \mid Z=1] - E[T \mid Z=0]}
$$

- Numerator: **ITT** — the effect the random assignment produced on the outcome.
- Denominator: **share of compliers** — the fraction the assignment actually moved.

Dividing one by the other gives the effect of treatment on a single complier. This is the Wald estimator — the simplest form of IV estimation.

In [4]:
num = df.loc[df["Z"] == 1, "Y"].mean() - df.loc[df["Z"] == 0, "Y"].mean()
den = df.loc[df["Z"] == 1, "T"].mean() - df.loc[df["Z"] == 0, "T"].mean()
wald = num / den
print(f"ITT (numerator):           {num:+.3f}")
print(f"Compliers share (denom):   {den:.3f}")
print(f"Wald estimator:            {wald:+.3f}")
print(f"True LATE:                 {true_effect:+.3f}")

ITT (numerator):           +2.634
Compliers share (denom):   0.588
Wald estimator:            +4.483
True LATE:                 +5.000


The Wald estimator lines up with the true effect (+5). The naive comparison was contaminated by type composition; the Wald formula isolates the part the random assignment actually moved and rescales it by the compliers' share.

In practice, the same estimate is computed via **2SLS (Two-Stage Least Squares)**: first regress $T$ on $Z$ to get fitted values $\hat T$, then regress $Y$ on $\hat T$.

In [5]:
import statsmodels.api as sm

# Stage 1: T ~ Z
first = sm.OLS(df["T"], sm.add_constant(df["Z"])).fit()
df["T_hat"] = first.predict(sm.add_constant(df["Z"]))

# Stage 2: Y ~ T_hat
second = sm.OLS(df["Y"], sm.add_constant(df["T_hat"])).fit()
print(f"2SLS estimated treatment effect: {second.params['T_hat']:+.3f}")

2SLS estimated treatment effect: +4.483


## 6) Three assumptions for a valid instrument

For the Wald formula to deliver a **causal** quantity, three conditions must hold simultaneously. If any one fails, the estimate is no longer causally interpretable.

**(a) Relevance** — $Z$ must actually move $T$.
$$\operatorname{Cov}(Z, T) \neq 0$$
If $Z$ barely moves $T$, the denominator approaches zero and the estimator explodes — the "weak instrument" problem.

**(b) Exclusion restriction** — $Z$ can affect $Y$ only through $T$.
If being assigned tablets changes school morale even when no tablets are used, exclusion is broken.

**(c) Independence** — $Z$ must be independent of the potential outcomes.
$$(Y_0, Y_1) \perp Z$$
In an RCT this is satisfied by design. For a random assignment produced by nature, you must argue the case by case.

As a DAG:

```{mermaid}
flowchart LR
    Z["Z<br/>(instrument)"] --> T["T<br/>(treatment)"]
    T --> Y["Y<br/>(outcome)"]
    U["U<br/>(unobserved confounder)"] -.-> T
    U -.-> Y

    classDef instrument fill:#fff3bf,stroke:#f59f00,color:#d9480f;
    classDef dashed stroke-dasharray: 5 5;
    class Z instrument;
    class U dashed;
```

$U$ is an unobserved variable that affects both $T$ and $Y$. A naive regression $Y \sim T$ is biased because of $U$. But if $Z$ is independent of $U$ and the only path from $Z$ to $Y$ is through $T$, then $Z$ isolates a "clean" slice of variation in $T$ that we can use to identify the $T \to Y$ effect.

## 7) National scale: the Vietnam draft lottery (Angrist 1990)

The United States, 1969. To draft people for the Vietnam War, the government used a lottery: balls labeled with all 365 birthdays were drawn bingo-style, and the lowest numbers went first.

Your birthday is something you can't pick, and the lottery number was unrelated to your education, occupation, or family. As close to a perfectly natural random assignment as you'll get.

The economist Joshua Angrist used this lottery as an instrument for the effect of **military service on lifetime earnings**.

- $Z$ : draft eligibility based on lottery number
- $T$ : whether the person actually served
- $Y$ : later earnings

A naive veteran-vs-non-veteran comparison conflates the effect of service with the selection of who chose to serve and who avoided it. Using the lottery as an instrument isolates the part chance moved.

Angrist (1990) estimated that military service reduced subsequent earnings of white male veterans by about **15%** — a result that overturns the casual "service = experience = higher earnings" intuition and remains one of the canonical IV findings.

## 8) Summary

- **Naive mean differences** are biased whenever there is unobserved confounding.
- **RCTs** break the bias by random assignment, but non-compliance reopens it for "actually treated" comparisons.
- **Instrumental variables (IV)** rescue identification by finding a variable $Z$ that moves $T$ but only affects $Y$ through $T$ — isolating the part driven by chance.
- **Wald estimator / 2SLS**: ITT divided by share of compliers. What you identify is **LATE** — the average treatment effect *for the compliers*.
- **Three locks**: relevance, exclusion, independence. All three must open or the IV door stays shut.

The hardest part of an IV analysis isn't the formula — it's **finding a plausible accident in the real world**. Snow's water companies, Angrist's lottery, and maybe your own birthday are all examples of accidents waiting to be used.

## References

This notebook draws on Angrist (1990), Imbens & Angrist (1994), Angrist & Pischke (2009), Snow (1855), and Matheus Facure's *Causal Inference for the Brave and True*.

- **Angrist, J. D. (1990)**. Lifetime Earnings and the Vietnam Era Draft Lottery: Evidence from Social Security Administrative Records. *American Economic Review*, 80(3), 313-336.

- **Imbens, G. W., & Angrist, J. D. (1994)**. Identification and Estimation of Local Average Treatment Effects. *Econometrica*, 62(2), 467-475.

- **Angrist, J. D., & Pischke, J.-S. (2009)**. *Mostly Harmless Econometrics: An Empiricist's Companion*. Princeton University Press.

- **Snow, J. (1855)**. *On the Mode of Communication of Cholera*. John Churchill.

- **Facure, M. (2022)**. 08 - Instrumental Variables. *Causal Inference for the Brave and True*.  
  [https://matheusfacure.github.io/python-causality-handbook/08-Instrumental-Variables.html](https://matheusfacure.github.io/python-causality-handbook/08-Instrumental-Variables.html)

- **Causal Studio**. IV in a Nutshell (EN).  
  [https://www.youtube.com/@CausalStudio](https://www.youtube.com/@CausalStudio)